<a href="https://colab.research.google.com/github/Rashmi-Sandeepani/Text-Detection-----NLP/blob/main/Text_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, re, time, unicodedata
from datetime import datetime
import pandas as pd
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.1-flash-lite"
print("Connected! Using model:", MODEL)

# **Step 1 - Connecting Gemini Model.**

In [ ]:
def ask_gemini(prompt, tries=4):
    for attempt in range(1, tries + 1):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            return response.text
        except Exception as error:
            if "503" in str(error) and attempt < tries:
                print("Gemini is busy, waiting", 2 ** attempt, "seconds...")
                time.sleep(2 ** attempt)
            else:
                return "Gemini error: " + str(error)

print(ask_gemini("Say hello in Tamil and English."))

# **Step 2 - Cleaning messy text before anything else touches it.**

In [ ]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    for fancy, plain in {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                         "\u2013": "-", "\u2014": "-", "\u2026": "..."}.items():
        text = text.replace(fancy, plain)
    text = re.sub(r"([!?])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print(clean_text("Hello     WORLD!!!   It’s  “great” — isn’t it???"))

# **Step 3 - Cutting the text into sentences.**

In [ ]:
ABBREVIATIONS = ["Dr.", "Mr.", "Mrs.", "Ms.", "Prof.", "St.", "e.g.", "i.e.",
                 "a.m.", "p.m.", "U.S.", "etc.", "vs.", "No."]

def split_sentences(text):
    for abbr in ABBREVIATIONS:
        text = text.replace(abbr, abbr.replace(".", "<DOT>"))
    text = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", text)
    pieces = re.split(r"(?<=[.!?])\s+(?=[\"'#A-Z0-9])", text)
    return [p.replace("<DOT>", ".").strip() for p in pieces if p.strip()]

for s in split_sentences("Dr. Silva paid $4.50 at 9 a.m. today. Was it worth it? Yes!"):
    print("-", s)

# **Step 4 - Counting characters, words and sentences.**

In [ ]:
def count_stats(text, sentences):
    return {"characters": len(text),
            "characters_no_spaces": len(text.replace(" ", "")),
            "words": len(text.split()),
            "sentences": len(sentences)}

t = "I love Python. It is fun!"
print(count_stats(t, split_sentences(t)))

# **Step 5 - Breaking text into word tokens and asking Gemini how many subword tokens it sees.**

In [ ]:
def word_tokens(text):
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text.lower())

def gemini_token_count(text):
    return client.models.count_tokens(model=MODEL, contents=text).total_tokens

sample = "I don't think tokenization is hard!"
print("Word tokens:", word_tokens(sample), len(word_tokens(sample)))
print("Gemini tokens:", gemini_token_count(sample))

# **Step 6 - Finding emails, dates, hashtags and phone numbers.**

In [ ]:
PATTERNS = {
    "emails":   r"[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}",
    "dates":    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{4}\b|\b\d{4}-\d{2}-\d{2}\b|\b\d{1,2} (?:January|February|March|April|May|June|July|August|September|October|November|December) \d{4}\b",
    "hashtags": r"(?<![\w#])#[A-Za-z]\w*",
    "phones":   r"(?<![\w+])(?:\+\d{1,3}[\s-]?|\(0\d{1,3}\)\s?|0)\d[\d\s-]{6,13}\d",
}

def find_patterns(text):
    return {name: re.findall(pattern, text) for name, pattern in PATTERNS.items()}

print(find_patterns("Email sanka@example.com on 21/11/2026, call +94 77 123 4567 #NLP"))

# **Step 7 - Gemini reads the text and gives insights, and it can answer questions about it.**

In [ ]:
def gemini_insights(text):
    prompt = f"""You are a helpful text analyst. Read the text below and reply with:
1. SUMMARY: a 2-sentence summary
2. MOOD: positive, negative or neutral, with one reason
3. KEYWORDS: the 5 most important words
4. QUESTIONS: 3 quiz questions about the text, with answers

Text:
{text}"""
    return ask_gemini(prompt)

def ask_about_text(text, question):
    prompt = f"""Answer the question using ONLY the text below.
If the answer is not in the text, say "The text does not say."

Text:
{text}

Question: {question}"""
    return ask_gemini(prompt)

story = "The Riverside Bridge opened on 14 February 2026. It cost 13.5 million dollars and cut travel time by 20 minutes."
print(gemini_insights(story))
print(ask_about_text(story, "How much did it cost?"))
print(ask_about_text(story, "What colour is the bridge?"))

# **Step 8 - Saving every investigated text into a library file and one big function that runs every above 6 tasks.**

In [ ]:
LIBRARY_FILE = "my_library.csv"

def save_to_library(text, source, stats):
    if os.path.exists(LIBRARY_FILE):
        next_number = len(pd.read_csv(LIBRARY_FILE)) + 1
    else:
        next_number = 1
    doc_id = f"DOC_{next_number:04d}"
    row = pd.DataFrame([{"doc_id": doc_id, "source": source,
                         "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
                         "words": stats["words"], "text": text}])
    row.to_csv(LIBRARY_FILE, mode="a", index=False,
               header=not os.path.exists(LIBRARY_FILE))
    return doc_id


def analyse(raw_text, source="typed by user"):
    text = clean_text(raw_text)                 # 🧽 Station 1
    sentences = split_sentences(text)           # ✂️ Station 2
    stats = count_stats(text, sentences)        # 🔢 Station 3
    words = word_tokens(text)                   # 🧩 Station 4
    found = find_patterns(text)                 # 🔎 Station 5
    insights = gemini_insights(text)            # 🧠 Station 6
    doc_id = save_to_library(text, source, stats)  # 📚 Station 7

    report = f"# Text Detective Report ({doc_id})\n\n## Counts\n"
    for name, value in stats.items():
        report += f"- {name}: {value}\n"
    report += "\n## Sentences\n"
    for i, s in enumerate(sentences, start=1):
        report += f"{i}. {s}\n"
    report += f"\n## Tokens\n- Word tokens: {len(words)}\n- Gemini tokens: {gemini_token_count(text)}\n"
    report += "\n## Patterns found\n"
    for name, items in found.items():
        report += f"- {name}: {', '.join(items) if items else 'none'}\n"
    report += "\n## Gemini's insights\n" + insights
    return report


test = """Hello   team!!! Dr. Perera’s meeting is on 21/11/2026 at 9.30 a.m. — please email
sanka@example.com or call +94 77 123 4567. #NLP #Gemini rocks… We paid $4.50 for coffee."""
print(analyse(test, source="test message"))

# **Step 9 - Making a simple ui.**

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def ask_tab(text, question):
    if not text.strip() or not question.strip():
        return "Please paste some text and type a question."
    return ask_about_text(clean_text(text), question)

with gr.Blocks(title="Text Detective") as app:
    gr.Markdown("# 🔍 Text Detective\nPaste any text and let the detective investigate it.")
    text_box = gr.Textbox(label="Your text", lines=8, placeholder="Paste your text here...")

    with gr.Tab("Analyse"):
        analyse_button = gr.Button("Investigate!")
        report_box = gr.Markdown()
        analyse_button.click(fn=analyse, inputs=text_box, outputs=report_box)

    with gr.Tab("Ask a question"):
        question_box = gr.Textbox(label="Your question")
        ask_button = gr.Button("Ask Gemini")
        answer_box = gr.Markdown()
        ask_button.click(fn=ask_tab, inputs=[text_box, question_box], outputs=answer_box)

app.launch()